# Unified Continuous Generative Models

为了详谈加速生成，我们愿意先建立统一的框架。为你介绍 UCGM，我们指出即使一致性模型或其他蒸馏模型与原始流匹配与扩散模型存在巨大的差异，他们之间仍然存在统一的理论框架，包括了目标函数以及训练与采样。这揭露了加速生成的一部分数学本质。

推荐你读 https://arxiv.org/abs/2505.07447 Unified Continuous Generative Models 这是 UCGM 原文。

# 统一框架

我们必须指出，UCGM 为了统一一致性模型与扩散模型框架付出了理论简洁性上的代价。但是这也许是必要的。同时，UCGM 整合了一些实用的训练或采样技巧，这使得其表现超越了过去的采样器与模型。

## 目标函数

我们直接给出统一框架下的目标函数形式
$$\mathcal{L}(\boldsymbol{\theta}) := \mathbb{E}_{(\mathbf{z}, \mathbf{x}) \sim p(\mathbf{z}, \mathbf{x}), t \sim \phi(t)} \left[ \frac{1}{\omega(t)} \| \boldsymbol{F}_{\boldsymbol{\theta}}(\mathbf{x}_t, t) - \mathbf{z}_t \|_2^2 \right]$$
其中时间步 $t \in [0, 1]$ 并且分布遵从一个特殊的概率密度函数 $\phi(t)$，$\omega(t)$ 是损失函数的权重，$\boldsymbol{F}_{\boldsymbol{\theta}}$ 是一个参数为 $\boldsymbol{\theta}$ 的神经网络，$\mathbf{x}_t = \alpha(t)\mathbf{z} + \gamma(t)\mathbf{x}$，且 $\mathbf{z}_t = \hat{\alpha}(t)\mathbf{z} + \hat{\gamma}(t)\mathbf{x}$。这里的 $\alpha(t)$，$\gamma(t)$，$\hat{\alpha}(t)$ 和 $\hat{\gamma}(t)$ 是 UCGM 定义的统一传输参数。

其中我们要求 $\alpha(t)$ 在区间 $t \in [0, 1]$ 上连续，且满足 $\alpha(0) = 0, \alpha(1) = 1$，以及 $\frac{\mathrm{d}\alpha(t)}{\mathrm{d}t} \geq 0$。$\gamma(t)$ 在区间 $t \in [0, 1]$ 上连续，且满足 $\gamma(0) = 1, \gamma(1) = 0$，以及 $\frac{\mathrm{d}\gamma(t)}{\mathrm{d}t} \leq 0$。对于所有 $t \in (0, 1)$，恒有 $|\alpha(t) \cdot \hat{\gamma}(t) - \hat{\alpha}(t) \cdot \gamma(t)| > 0$。

为了便于解释，我们基于模型 $\boldsymbol{F}_{\boldsymbol{\theta}}$ 定义两个预测函数 $$\boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_t, \mathbf{x}_t, t) := \frac{\alpha(t) \cdot \boldsymbol{F}_t - \hat{\alpha}(t) \cdot \mathbf{x}_t}{\alpha(t) \cdot \hat{\gamma}(t) - \hat{\alpha}(t) \cdot \gamma(t)} \quad \& \quad \boldsymbol{f}^{\mathbf{z}}(\boldsymbol{F}_t, \mathbf{x}_t, t) := \frac{\hat{\gamma}(t) \cdot \mathbf{x}_t - \gamma(t) \cdot \boldsymbol{F}_t}{\alpha(t) \cdot \hat{\gamma}(t) - \hat{\alpha}(t) \cdot \gamma(t)}$$

其中我们定义 $\boldsymbol{F}_t := \boldsymbol{F}_{\boldsymbol{\theta}}(\mathbf{x}_t, t)$。原始训练目标因此可以变为 $$\mathcal{L}(\boldsymbol{\theta}) = \mathbb{E}_{(\mathbf{z}, \mathbf{x}) \sim p(\mathbf{z}, \mathbf{x}), t \sim \phi(t)} \left[ \frac{1}{\hat{\omega}(t)} \| \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_{\boldsymbol{\theta}}(\mathbf{x}_t, t), \mathbf{x}_t, t) - \mathbf{x} \|_2^2 \right]$$

我们不赘述此处的变换过程，这是一个简单的代换。为了与我们的原始训练目标保持梯度一致，我们定义一个新的权重函数 $\hat{\omega}(t)$ 为 $\hat{\omega}(t) := \frac{\alpha(t) \cdot \alpha(t) \cdot \omega(t)}{(\alpha(t) \cdot \hat{\gamma}(t) - \hat{\alpha}(t) \cdot \gamma(t))^2}$。并且为了将少步生成模型与多步生成模型统一起来，引入一致性比例 $\lambda \in [0, 1]$。

现在我们再给出最终的目标函数版本 
$$\mathcal{L}(\boldsymbol{\theta}) = \mathbb{E}_{(\mathbf{z}, \mathbf{x}) \sim p(\mathbf{z}, \mathbf{x}), t \sim \phi(t)} \left[ \frac{1}{\hat{\omega}(t)} \| \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_{\boldsymbol{\theta}}(\mathbf{x}_t, t), \mathbf{x}_t, t) - \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_{\boldsymbol{\theta}^-}(\mathbf{x}_{\lambda t}, \lambda t), \mathbf{x}_{\lambda t}, \lambda t) \|_2^2 \right] \quad (*)$$
其中 $\theta^-$ 是指 EMA 或者其他形式非实时权重。

$(*)$ 是我们得到最终目标函数形式。其与之前的形式是统一的，因为 $$f^\mathbf{x}(F_{\theta^-}(x_{\lambda t}, \lambda t), x_{\lambda t}, \lambda t) \bigg|_{\lambda=0} = f^\mathbf{x}(F_{\theta^-}(x_0, 0), x_0, 0) = x$$
因此 $(*)$ 是一个更加广义的形式。

在实际工程计算当中，我们选取已经广泛验证有效的权重 $\hat{\omega}(t) = \frac{\tan(t)}{4}$ 以及其他参数形式。首先我们定义一个一阶差分近似 $$\Delta f_t^x := \frac{f^x(F_{\theta}(x_t, t), x_t, t) - f^x(F_{\theta^-}(x_{\lambda t}, \lambda t), x_{\lambda t}, \lambda t)}{t - \lambda t}$$

下面给出实际目标函数 $$\mathcal{L}(\theta) = \mathbb{E}_{(\mathbf{z}, \mathbf{x}) \sim p(\mathbf{z}, \mathbf{x}), t \sim \phi(t)} \left[ \cos(t) \left\| F_{\theta}(x_t, t) - \left( F_{\theta^-}(x_t, t) + \frac{4\alpha(t)\Delta f_t^x}{\sin(t) \cdot (\alpha(t)\hat{\gamma}(t) - \hat{\alpha}(t)\gamma(t))} \right) \right\|^2_2 \right]$$

上式是实际训练采样所使用的目标函数，我们仅仅只是对目标函数 $(*)$ 做简单的代换。

不过一个问题是当 $\lambda \to 1$，一阶差分 $\Delta f_t^x$ 在计算机实际精度下会有问题，因为 BF16 在如此细微的计算时会产生不稳定的梯度。为此我们做出简单改进 $$\Delta \boldsymbol{f}_t^{\mathbf{x}} = \frac{1}{2\epsilon} \left( \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_{\boldsymbol{\theta}^-}(\mathbf{x}_{t+\epsilon}, t+\epsilon), \mathbf{x}_{t+\epsilon}, t+\epsilon) - \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_{\boldsymbol{\theta}^-}(\mathbf{x}_{t-\epsilon}, t-\epsilon), \mathbf{x}_{t-\epsilon}, t-\epsilon) \right)$$
这是一个简单的估计方法，其中 $\epsilon$ 被设置为 $0.005$。更多的，我们为这个差分裁剪到 $[-1, 1]$ 以稳定训练。

现在我们来说说时间步 $t \sim \phi(t)$ 的分布。我们选取概率密度函数 $\phi(t)$ 为 $\text{Beta}(\theta_1, \theta_2)$。更详细的 
$$\text{Beta}(x, y) = \int_0^1 t^{x-1}(1-t)^{y-1} dt$$
其中 $x, y > 0$。

最后一个技巧被称为 Learning Enhanced Target。简而言之就是当我们使用 CFG，在推理时我们必须使用模型推理两遍，这意味着模型推理步数的直接翻倍。这是因为 $$\mathbf{\hat{s}} = \mathbf{s}_\theta(\mathbf{x}, \emptyset) + \omega \left( \mathbf{s}_\theta(\mathbf{x}, y) - \mathbf{s}_\theta(\mathbf{x}, \emptyset) \right)$$
当 $\omega > 1$，有条件推理一次与无条件推理一次是必要的。但是如果我们将 $\omega = 1$，那么我们就仅仅需要使用有条件推理一次。为了保证仅仅推理一次就可以产出高质量的效果，我们让模型在训练阶段就直接去拟合外推后的结果

我们不再让模型去学习原始的 $x$ 和 $z$，而是学习一个增强后的目标 $x^*$ 和 $z^*$。并且我们设置一个时间分段点 $s$，原因是我们认为时间步 $t$ 较小时噪声比较轻微，此时模型直接学习外推结果是可行的。然而当时间步 $t$ 较大时，噪声变得浓厚，我们选择让模型学习一个预测目标与真实目标的均值。这是更保守的学习策略。

更具体的，对于 $t \in [0, s]$，令 
$$\mathbf{x}^{\star} = \mathbf{x} + \zeta \cdot \left( \text{sg}(\boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_t, \mathbf{x}_t, t)) - \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_t^{\varnothing}, \mathbf{x}_t, t) \right)$$
和
$$\mathbf{z}^{\star} = \mathbf{z} + \zeta \cdot \left( \text{sg}(\boldsymbol{f}^{\mathbf{z}}(\boldsymbol{F}_t, \mathbf{x}_t, t)) - \boldsymbol{f}^{\mathbf{z}}(\boldsymbol{F}_t^{\varnothing}, \mathbf{x}_t, t) \right)$$

这里 $\boldsymbol{F}_t^{\varnothing} = \boldsymbol{F}_{\boldsymbol{\theta}^-}(\mathbf{x}_t, t, \varnothing)$ 且 $\boldsymbol{F}_t = \boldsymbol{F}_{\boldsymbol{\theta}^-}(\mathbf{x}_t, t)$。

而对于 $t \in [s, 1]$，令 $\mathbf{x}^{\star} = \mathbf{x} + \frac{1}{2}(\boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_t, \mathbf{x}_t, t) - \mathbf{x})$ 和 $\mathbf{z}^{\star} = \mathbf{z} + \frac{1}{2}(\boldsymbol{f}^{\mathbf{z}}(\boldsymbol{F}_t, \mathbf{x}_t, t) - \mathbf{z})$。我们统一设定 $s = 0.75$。

所以我们实际上选择在训练时付出更高的算力代价，换取推理时翻倍的效率。请注意梯度停止算子 $sg$ 很关键，这意味着我们将其视为常数，这是为了防止模型目标偏移的学习崩溃。

## 训练

我为你讲述 UCGM 框架下如何做训练。实际上我们要分情况讨论 $\lambda$ 是否为 $1$，因为涉及到我们上述所说的计算精度问题。

假设我们已有数据集 $D$，预设的传输参数 $\alpha(t), \gamma(t), \hat{\alpha}(t),\hat{\gamma}(t)$，增强系数 $\zeta$，时间步密度函数 $\text{Beta}(\theta_1, \theta_2)$，以及更多基本的机器学习参数。

首先采样噪声 $z \sim \mathcal{N}(0, \mathbf{I})$ 与真实数据样本 $x \sim D$ 以及时间步 $t \sim \text{Beta}(\theta_1, \theta_2)$。

向前加噪得到 $x_t = \alpha(t) z + \gamma(t) x, x_{\lambda t}= \alpha(\lambda t)z + \gamma(\lambda t)x$。

输入模型得到 $F_t = F_{\theta}(x_t, t)$。初始化 $z^* = z, x^* =x$。

如果我们使用内化 CFG 方案，我们按照先前所说重新设置 $z^*, x^*$。

并且重新计算加噪 $x_t^* = \alpha(t) z^* + \gamma(t) x^*, x_{\lambda t}^*= \alpha(\lambda t)z^* + \gamma(\lambda t)x^*$ 或者 $x_{t+\epsilon}^* = \alpha(t+\epsilon) z^* + \gamma(t+\epsilon) x^*, x_{t-\epsilon}^*= \alpha(t-\epsilon)z^* + \gamma(t-\epsilon)x^*$。

最后计算一阶差分，并且计算最终损失函数。遍历一个 Batch 之后计算平均损失，反向传播更新梯度。

以下是完整的训练算法。

$$\begin{array}{l}
\hline
\textbf{Algorithm 1 (UCGM-T). } \text{A Unified and Efficient Trainer for Few-step and Multi-step Continuous} \\
\text{Generative Models (including Diffusion, Flow Matching, and Consistency Models)} \\
\hline
\textbf{Require: } \text{Dataset } D, \text{transport coefficients } \{\alpha(\cdot), \gamma(\cdot), \hat{\alpha}(\cdot), \hat{\gamma}(\cdot)\}, \text{neural network } \boldsymbol{F}_{\boldsymbol{\theta}}, \text{enhancement} \\
\quad \text{ratio } \zeta, \text{Beta distribution parameters } (\theta_1, \theta_2), \text{learning rate } \eta, \text{stop gradient operator } \color{red}{\text{sg}}. \\
\textbf{Ensure: } \text{Trained neural network } \boldsymbol{F}_{\boldsymbol{\theta}} \text{ for generating samples from } p(\mathbf{x}). \\
1: \textbf{repeat} \\
2: \quad \text{Sample } \mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I}), \mathbf{x} \sim D, t \sim \phi(t) := \text{Beta}(\theta_1, \theta_2) \\
3: \quad \text{Compute input data, such as } \mathbf{x}_t = \alpha(t) \cdot \mathbf{z} + \gamma(t) \cdot \mathbf{x} \text{ and } \mathbf{x}_{\lambda t} = \alpha(\lambda t) \cdot \mathbf{z} + \gamma(\lambda t) \cdot \mathbf{x} \\
4: \quad \text{Compute model output } \boldsymbol{F}_t = \boldsymbol{F}_{\boldsymbol{\theta}}(\mathbf{x}_t, t) \text{ and set } \mathbf{z}^{\star} = \mathbf{z} \text{ and } \mathbf{x}^{\star} = \mathbf{x} \\
5: \quad \textbf{if } \zeta \in (0, 1) \textbf{ then} \\
6: \quad \quad \text{Let } \boldsymbol{F}_t^{\varnothing} = \boldsymbol{F}_{\boldsymbol{\theta}^-}(\mathbf{x}_t, t, \varnothing) \text{ to get enhanced } \mathbf{x}^{\star} = \boldsymbol{\xi}(\mathbf{x}, t, \boldsymbol{f}^{\mathbf{x}}(\color{red}{\text{sg}}(\boldsymbol{F}_t), \mathbf{x}_t, t), \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_t^{\varnothing}, \mathbf{x}_t, t)) \\
\quad \quad \text{and } \mathbf{z}^{\star} = \boldsymbol{\xi}(\mathbf{z}, t, \boldsymbol{f}^{\mathbf{z}}(\color{red}{\text{sg}}(\boldsymbol{F}_t), \mathbf{x}_t, t), \boldsymbol{f}^{\mathbf{z}}(\boldsymbol{F}_t^{\varnothing}, \mathbf{x}_t, t)) \{ \text{Note that } \color{green}{\boldsymbol{\xi}(\mathbf{a}, t, \mathbf{b}, \mathbf{d}) := \mathbf{a} +} \\
\quad \quad \color{green}{(\zeta + \mathbf{1}_{t>s}(\frac{1}{2} - \zeta)) \cdot (\mathbf{b} - \mathbf{1}_{t>s} \cdot \mathbf{a} - \mathbf{d}(1 - \mathbf{1}_{t>s})), \text{ where } \mathbf{1}(\cdot) \text{ is the indicator function} } \} \\
7: \quad \textbf{end if} \\
8: \quad \textbf{if } \lambda \in [0, 1) \textbf{ then} \\
9: \quad \quad \text{Compute } \mathbf{x}_t^{\star} = \alpha(t) \cdot \mathbf{z}^{\star} + \gamma(t) \cdot \mathbf{x}^{\star} \text{ and } \mathbf{x}_{\lambda t}^{\star} = \alpha(\lambda t) \cdot \mathbf{z}^{\star} + \gamma(\lambda t) \cdot \mathbf{x}^{\star} \\
10: \quad \quad \text{Compute } \Delta \boldsymbol{f}_t^{\mathbf{x}} = \boldsymbol{f}^{\mathbf{x}}(\color{red}{\text{sg}}(\boldsymbol{F}_t), \mathbf{x}_t^{\star}, t) \cdot (\frac{1}{t - \lambda t}) - \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_{\boldsymbol{\theta}^-}(\mathbf{x}_{\lambda t}, \lambda t), \mathbf{x}_{\lambda t}^{\star}, \lambda t) \cdot (\frac{1}{t - \lambda t}) \{ \text{Note} \\
\quad \quad \color{green}{\text{that for } \lambda = 0, \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_{\boldsymbol{\theta}^-}(\mathbf{x}_0, 0), \mathbf{x}_0^{\star}, 0) = \mathbf{x}^{\star} } \} \\
11: \quad \textbf{else if } \lambda = 1 \textbf{ then} \\
12: \quad \quad \text{Compute } \mathbf{x}_{t+\epsilon}^{\star} = \alpha(t+\epsilon) \cdot \mathbf{z}^{\star} + \gamma(t+\epsilon) \cdot \mathbf{x}^{\star} \text{ and } \mathbf{x}_{t-\epsilon}^{\star} = \alpha(t-\epsilon) \cdot \mathbf{z}^{\star} + \gamma(t-\epsilon) \cdot \mathbf{x}^{\star} \\
13: \quad \quad \text{Let } \Delta \boldsymbol{f}_t^{\mathbf{x}} = \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_{\boldsymbol{\theta}^-}(\mathbf{x}_{t+\epsilon}, t+\epsilon), \mathbf{x}_{t+\epsilon}^{\star}, t+\epsilon) \cdot (\frac{1}{2\epsilon}) - \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_{\boldsymbol{\theta}^-}(\mathbf{x}_{t-\epsilon}, t-\epsilon), \mathbf{x}_{t-\epsilon}^{\star}, t-\epsilon) \cdot (\frac{1}{2\epsilon}) \\
14: \quad \textbf{end if} \\
15: \quad \text{Compute } \boldsymbol{F}_t^{\text{target}} = \color{red}{\text{sg}}(\boldsymbol{F}_t) - \frac{4\alpha(t)}{\alpha(t)\hat{\gamma}(t) - \hat{\alpha}(t)\gamma(t)} \cdot \frac{\text{clip}(\Delta \boldsymbol{f}_t^{\mathbf{x}}, -1, 1)}{\sin(t)} \\
16: \quad \text{Compute loss } \mathcal{L}_t(\boldsymbol{\theta}) = \cos(t) \left\| \boldsymbol{F}_t - \boldsymbol{F}_t^{\text{target}} \right\|_2^2 \text{ and update } \boldsymbol{\theta} \leftarrow \boldsymbol{\theta} - \eta \nabla_{\boldsymbol{\theta}} \int_0^1 \phi(t) \mathcal{L}_t(\boldsymbol{\theta}) \mathrm{d}t \\
17: \textbf{until } \text{Convergence} \\
\hline
\end{array}$$

## 推理

我们指出一件事，无论是扩散模型还是一致性模型，推理实际上一直在重复分解到重构的过程。具体来说，模型根据带噪声图像推理出原始图像，再重新前向加噪到下一个带噪声图像，重复这个过程直至时间步 $t$ 为终点。

符号化的表述是，给定时间步 $t$ 下带噪图像 $\tilde{x}_t$，模型输出 $F_t = F_{\theta^-}(\tilde{x}_t, t)$，计算 $\hat{x}_t = f^{\mathbf{x}}(F_t, \tilde{x}_t, t)$ 以及 $\hat{z}_t = f^{\mathbf{x}}(F_t, \tilde{x}_t, t)$。

这意味着带噪图像被分解为 $\tilde{x}_t = \alpha(t) \hat{z}_t + \gamma(t) \hat{x}_t$。

下一步是根据选定的下一时间步 $t'$ 重构下一带噪图像 $\tilde{x}_{t'} = \alpha(t') \hat{z}_t + \gamma(t') \hat{x}_t$。

现在重复整个过程，直至时间步抵达尽头。

更多的，我们指出以上整个过程中可以优化的两个技巧。第一件事是，模型根据一步时间步带噪图像还原出的原始图像误差非常大，这意味着每一步重构过程都在累计误差。受到 CFG 向量外推的启发，我们指出一个技巧 Extrapolating the estimation。

定义外推系数 $\kappa \in [0,1]$，对于推理重构结果做向量外推作为新的重构结果。换句话说 $\tilde{x}_{t'} \leftarrow \tilde{x}_{t'} + \kappa (\tilde{x}_{t'} - \tilde{x}_{t})$ 以及 $\tilde{z}_{t'} \leftarrow \tilde{z}_{t'} + \kappa (\tilde{z}_{t'} - \tilde{z}_{t})$。这实际上是说，我们将重构结果改为原重构结果与上一步带噪结果做线性插值，某种意义上我们减少了更新步长以保证更加柔和的更新过程。

第二件事则是 Langevin Dynamics，我们为了生成多样性加入随机采样的噪声项。定义随机系数 $\rho$，我们重新定义重构过程
$$\tilde{x}_{t'} = \alpha(t')(\sqrt{1 - \rho} \cdot \hat{z}_t + \sqrt{\rho} \cdot z) + \gamma(t') \hat{x}_t $$
其中 $z \sim \mathcal{N}(0 , \mathbf{I})$。原作者推荐 $\rho = \lambda$，不要忘记 $\lambda$ 系数是我们在目标函数小节所说的一致性系数。

最后我们来说说采样器的阶数。简单来说我们根据 ODE 每点方向求解的方式。过去的几章中，我们实际上仅仅提到了一阶求解器，也就是 Euler 求解器。我们根据模型在当前点预测的方向更新到下一点。

但是实际上我们还有二阶求解释甚至更加高阶求解器。我们详细介绍一下二阶求解器，如 Heun 求解器。假设我们求解
$$\frac{dy}{dt} = f(t, y), \quad y(t_0) = y_0$$
我们尝试通过 $y_n$ 估算下一个时间点 $t_{n+1} = t_n + h$ 的状态 $y_{n+1}$。首先做一步标准 Euler 法
$$\tilde{y}_{n+1} = y_n + h \cdot f(t_n, y_n)$$
再计算预估终点 $\tilde{y}_{n+1}$ 处的斜率，然后取两点斜率的平均值，作为最终步进方向 
$$y_{n+1} = y_n + \frac{h}{2} \left[ f(t_n, y_n) + f(t_{n+1}, \tilde{y}_{n+1}) \right]$$
二阶求解器精度更高，但是代价是更新一步需要运用两次模型输出。因此对于二阶求解器，我们直接将预设最大步数 $N$ 减半为 $\lfloor (N+1)/2 \rfloor$，这样可以兼顾效率与精度。

下面给出完整的训练算法。

$$\begin{array}{l}
\hline
\textbf{Algorithm 2 (UCGM-S). } \text{A Unified and Efficient Sampler for Few-step and Multi-step Continuous} \\
\text{Generative Models (including Diffusion, Flow Matching, and Consistency Models)} \\
\hline
\textbf{Require: } \text{Initial } \tilde{\mathbf{x}} \sim \mathcal{N}(\mathbf{0}, \mathbf{I}), \text{transport coefficients } \{\alpha(\cdot), \gamma(\cdot), \hat{\alpha}(\cdot), \hat{\gamma}(\cdot)\}, \text{trained model } \boldsymbol{F}_{\boldsymbol{\theta}}, \\
\quad \text{sampling steps } N, \text{order } \nu \in \{1, 2\}, \text{time schedule } \mathcal{T}, \text{extrapolation ratio } \kappa, \text{stochastic ratio } \rho. \\
\textbf{Ensure: } \text{Final generated sample } \tilde{\mathbf{x}} \sim p(\mathbf{x}) \text{ and history samples } \{\hat{\mathbf{x}}_i\}_{i=0}^N \text{ over generation process.} \\
1: \text{Let } N \leftarrow \lfloor (N + 1) / 2 \rfloor \text{ if using second order sampling } (\nu = 2) \text{ } \{ \color{green}{\text{Adjusts total steps to match}} \\
\quad \color{green}{\text{first-order evaluation count}} \} \\
2: \textbf{for } i = 0 \text{ to } N - 1 \textbf{ do} \\
3: \quad \text{Compute model output } \boldsymbol{F} = \boldsymbol{F}_{\boldsymbol{\theta}^-}(\tilde{\mathbf{x}}, t_i), \text{ and then } \hat{\mathbf{x}}_i = \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}, \tilde{\mathbf{x}}, t_i) \text{ and } \hat{\mathbf{z}}_i = \boldsymbol{f}^{\mathbf{z}}(\boldsymbol{F}, \tilde{\mathbf{x}}, t_i) \\
4: \quad \textbf{if } i \geq 1 \textbf{ then} \\
5: \quad \quad \text{Compute extrapolated estimation } \hat{\mathbf{z}} = \hat{\mathbf{z}}_i + \kappa \cdot (\hat{\mathbf{z}}_i - \hat{\mathbf{z}}_{i-1}) \text{ and } \hat{\mathbf{x}} = \hat{\mathbf{x}}_i + \kappa \cdot (\hat{\mathbf{x}}_i - \hat{\mathbf{x}}_{i-1}) \\
6: \quad \textbf{end if} \\
7: \quad \text{Sample } \mathbf{z} \sim \mathcal{N}(\mathbf{0}, \mathbf{I}) \text{ } \{ \color{green}{\text{An example choice of } \rho \text{ for performing SDE-similar sampling is:}} \\
\quad \color{green}{\rho = \text{clip}(\frac{|t_i - t_{i+1}| \cdot 2\alpha(t_i)}{\alpha(t_{i+1})}, 0, 1)} \} \\
8: \quad \text{Compute estimated next time sample } \mathbf{x}' = \alpha(t_{i+1}) \cdot (\sqrt{1 - \rho} \cdot \hat{\mathbf{z}} + \sqrt{\rho} \cdot \mathbf{z}) + \gamma(t_{i+1}) \cdot \hat{\mathbf{x}} \\
9: \quad \textbf{if } \text{order } \nu = 2 \textbf{ and } i < N - 1 \textbf{ then} \\
10: \quad \quad \text{Compute prediction } \boldsymbol{F}' = \boldsymbol{F}_{\boldsymbol{\theta}}(\mathbf{x}', t_{i+1}), \hat{\mathbf{x}}' = \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}', \mathbf{x}', t_{i+1}) \text{ and } \hat{\mathbf{z}}' = \boldsymbol{f}^{\mathbf{z}}(\boldsymbol{F}', \mathbf{x}', t_{i+1}) \\
11: \quad \quad \text{Compute corrected next time sample } \mathbf{x}' = \tilde{\mathbf{x}} \cdot \frac{\gamma(t_{i+1})}{\gamma(t_i)} + \left( \alpha(t_{i+1}) - \frac{\gamma(t_{i+1})\alpha(t_i)}{\gamma(t_i)} \right) \cdot \frac{\hat{\mathbf{x}} + \hat{\mathbf{x}}'}{2} \\
12: \quad \textbf{end if} \\
13: \quad \text{Reset } \tilde{\mathbf{x}} \leftarrow \mathbf{x}' \\
14: \textbf{end for} \\
\hline
\end{array}$$

以上是 UCGM 的全体框架。总的来说，UCGM 不仅仅提出一个统一的训练与推理框架，还为原始的框架加上了诸多优化技巧。

在真实的测试数据集环境下，UCGM 的采样器配合其他模型可以将推理速度加速 $5 \times \sim 10 \times$，比如将 DiT-XL/2 的推理步数从 $500$ 步减少到 $100$ 步并且保持了 FID 水平。这是巨大的突破。

# 对应

最后我们来说说 UCGM 框架下对应到具体的 Score-based Models, Flow Matching Models 与 Consistency Models。这一对应的关键在于调节一致性系数 $\lambda$。

回顾我们的最终目标函数
$$\mathcal{L}(\boldsymbol{\theta}) = \mathbb{E}_{(\mathbf{z}, \mathbf{x}) \sim p(\mathbf{z}, \mathbf{x}), t \sim \phi(t)} \left[ \frac{1}{\hat{\omega}(t)} \| \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_{\boldsymbol{\theta}}(\mathbf{x}_t, t), \mathbf{x}_t, t) - \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_{\boldsymbol{\theta}^-}(\mathbf{x}_{\lambda t}, \lambda t), \mathbf{x}_{\lambda t}, \lambda t) \|_2^2 \right] \quad (*)$$

我们取 $\lambda = 0$，函数退化为 $$\mathcal{L}(\boldsymbol{\theta}) = \mathbb{E}_{(\mathbf{z}, \mathbf{x}) \sim p(\mathbf{z}, \mathbf{x}), t \sim \phi(t)} \left[ \frac{1}{\hat{\omega}(t)} \| \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_{\boldsymbol{\theta}}(\mathbf{x}_t, t), \mathbf{x}_t, t) - \mathbf{x} \|_2^2 \right]$$
这就是扩散模型的目标函数。因为 Score-based Models 的目标函数是预测噪声，在已经目前带噪图像情况下，这完全等价预测原始图像。

对于 Flow Matching 模型，我们取传输参数 $\alpha(t) = 1- t, \gamma(t) = t, \hat{\alpha}(t) = -1, \hat{\gamma}(t) = 1$，这就是标准的最优传输路径。

对应的，$\alpha(t) = t, \gamma(t) = 1, \hat{\alpha}(t) = 0, \hat{\gamma}(t) = 1$，这就是标准 VE 扩散模型路线。

我们取 $\lambda t = t - \Delta t$，函数退化为
$$\mathcal{L}(\boldsymbol{\theta}) = \mathbb{E}_{(\mathbf{z}, \mathbf{x}) \sim p(\mathbf{z}, \mathbf{x}), t \sim \phi(t)} \left[ \frac{1}{\hat{\omega}(t)} \| \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_{\boldsymbol{\theta}}(\mathbf{x}_t, t), \mathbf{x}_t, t) - \boldsymbol{f}^{\mathbf{x}}(\boldsymbol{F}_{\boldsymbol{\theta}^-}(\mathbf{x}_{t - \Delta t}, t - \Delta t), \mathbf{x}_{t - \Delta t}, t - \Delta t) \|_2^2 \right]$$
这就是一致性模型的目标函数。因为一致性函数希望任意两点之间预测的终点图像是一样的。更多的，这里解释了为什么预测项使用 EMA 权重 $\theta ^ -$，因为一致性函数要求了目标函数不可以使用实时权重，以防止模型自己学习自己的崩溃。

设置传输参数 $\hat{\alpha}(t) = 0, \hat{\gamma}(t) = 1$，现在变成了标准一致性模型。

在 $\lambda \in (0,1)$ 时，情况变得有趣。模型训练时不仅仅在尝试学习原始图像的映射，同时还受到一个隐式的一致性约束。或者我们观察展开形式或更直接地发现到这一点
$$\mathcal{L}(\theta) = \mathbb{E}_{(\mathbf{z}, \mathbf{x}) \sim p(\mathbf{z}, \mathbf{x}), t \sim \phi(t)} \left[ \cos(t) \left\| F_{\theta}(x_t, t) - \left( F_{\theta^-}(x_t, t) + \frac{4\alpha(t)\Delta f_t^x}{\sin(t) \cdot (\alpha(t)\hat{\gamma}(t) - \hat{\alpha}(t)\gamma(t))} \right) \right\|^2_2 \right]$$
这个目标函数要求了模型必须保证一定程度上的一致性，因为一阶差分也被计入了损失函数 MSE 距离之中。在同一个 PF ODE 轨迹上模型必须给出类似的终点映射结果。这某种意义上保证了模型学习的路线足够一致且直接。

这里其实蕴含了一种思想，也就是 Consitency Flow Matching。我们为原始 FM 模型损失函数加上一个一致性正则项，保证模型在首次训练就达到接近 Reflow 后训练效果。后续我们也许会介绍。

# 总结

本章我们介绍了一致性模型与扩散模型的统一框架 UCGM，这为之后的加速生成理论介绍打下坚实基础。

下一章我为你介绍 Meanflow，一种对于 Flow Matching 的改进技术，来自何恺明团队。